In [3]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [4]:
transform=transforms.ToTensor() # convert image into tensor 
train_data=datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_data=datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

In [5]:
train_loader=DataLoader(train_data,batch_size=32)
test_loader=DataLoader(test_data,batch_size=32)

### CNN

In [6]:
import torch.nn as  nn
import torch.optim as optim

In [52]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layer=nn.Sequential(
            # frist layer
            nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(32*14*14,256),
            nn.ReLU(),
            nn.Linear(256,10) 
        )
    def forward(self,x):
            x=self.conv_layer(x)
            x=x.view(x.size(0),-1)
            x=self.fc_layers(x)
            return x

In [53]:
model=CNN()
critenion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

#### Traninig the CNN

In [54]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for image ,labels in train_loader:
        optimizer.zero_grad()
        output=model.forward(image)
        loss=critenion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()
    print(f"epoch={epoch+1} and loss={epoch_training_loss/len(train_loader)}")
    

epoch=1 and loss=0.17162885118983687
epoch=2 and loss=0.05815990460962833
epoch=3 and loss=0.033403960114887256
epoch=4 and loss=0.019503214757889024
epoch=5 and loss=0.013135389505967305
epoch=6 and loss=0.010715019197507263
epoch=7 and loss=0.0074577607918413225
epoch=8 and loss=0.006527328501784145
epoch=9 and loss=0.004897196700846707
epoch=10 and loss=0.004453207062565673


### Evaluation of Model

In [70]:
import torch

correct_labels = 0
total_labels = 0

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)   # ✅ better practice
        
        _, predicted = torch.max(outputs, 1)
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)   # ✅ IMPORTANT
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
accuracy = (correct_labels / total_labels) * 100
print(f"Accuracy = {accuracy:.2f}%")

Accuracy = 98.28%


In [73]:
from sklearn.metrics import confusion_matrix,classification_report

In [72]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[ 977    1    0    0    0    0    1    1    0    0]
 [   0 1130    1    0    2    0    2    0    0    0]
 [   2    3 1018    0    1    0    1    6    1    0]
 [   0    0    4  990    0    7    0    3    0    6]
 [   1    0    0    0  971    0    4    0    0    6]
 [   2    0    1    3    0  880    3    0    2    1]
 [   6    3    0    0    1    4  944    0    0    0]
 [   1    3   11    1    0    0    0 1004    2    6]
 [   5    1    5    3    4    5    3    5  935    8]
 [   2    2    1    2    9    5    0    7    2  979]]


In [75]:
cm=classification_report(all_labels,all_preds)
print(cm)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       980
           1       0.99      1.00      0.99      1135
           2       0.98      0.99      0.98      1032
           3       0.99      0.98      0.99      1010
           4       0.98      0.99      0.99       982
           5       0.98      0.99      0.98       892
           6       0.99      0.99      0.99       958
           7       0.98      0.98      0.98      1028
           8       0.99      0.96      0.98       974
           9       0.97      0.97      0.97      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000

